# ζ(2): the CMF walk and its closed form

This notebook follows the supplied paper's **Theorem 6.2 (CMF Theorem 2, `thm:cmf2`)**, **Lemma 6.17 (`lem:z2-connection`)**, **Lemma 6.18 (`lem:z2-limits`)**, and Appendix `app:z2-blocks`.

Run every cell in a fresh Python kernel. The helper `companion.py` is included beside this notebook; the requirements are in the bundle README. All matrix and polynomial equalities below use exact rational arithmetic. RamanujanTools independently constructs the unit moves. The numerical approximation table is an illustration, not a proof of convergence.

**Scope.** We reproduce the finite readout, the four-block path, closed-form identities at specified depths, and the supplied finite generator certificate. The uniform identities and convergence arguments are the paper's analytic/algebraic proofs. The full rational identity of Lemma 6.17 is reconstructed directly from the paper's parameters using RamanujanTools and is verified symbolically below, together with its initial matrix and limit pairing. No Lean code is written, executed, or modified.

In [1]:
from pathlib import Path
import sys
# Works from this notebook folder or from the unzipped bundle root.
HERE = next((p.resolve() for p in [Path.cwd(), Path.cwd()/"zeta2"]
             if (p/"companion.py").is_file()), None)
if HERE is None:
    raise FileNotFoundError("Run this notebook from the zeta2 folder or bundle root.")
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE/"certificates"))
from fractions import Fraction as Q
from IPython.display import display, Markdown
import companion as z2
print("Local Python support loaded. No other notebook needs to be run.")

Local Python support loaded. No other notebook needs to be run.


## 1. Use the paper's parameter order

Throughout this companion,
\[
s_0=(0,1,1,1;1,1,1),\qquad t_0=(-11,10,9,11;2,1,3).
\]
The source repository uses the upper order $(x_1,x_2,x_3,x_0)$, so its start is $(1,1,1,0;1,1,1)$. These specify the same ${}_4F_3$ function, because its upper parameters are symmetric. `companion.py` exposes the paper's order; the preserved archival certificates retain their original order.

Write $\mathcal F_n=(F_n,\theta F_n,\theta^2F_n)$ as a **row** state. Products act on the right, and $\Phi_M(0)=I$. The initial rows are
\[
\rho_q=(2,0,0),\qquad \rho_p=(0,2,-1),\quad
q_n=\rho_q\Phi_M(n)e_0,\quad p_n=\rho_p\Phi_M(n)e_0.
\]
In the Python helper the readout matrix stores the $p$ row first, then the $q$ row. It returns `(p_n, q_n)` explicitly.

In [2]:
assert z2.START == (0,1,1,1,1,1,1)
assert z2.DIRECTION == (-11,10,9,11,2,1,3)
assert z2.matrix_readout(0) == (0,2)
print("s_0 =", z2.START)
print("t_0 =", z2.DIRECTION)
print("(p_0,q_0) =", z2.matrix_readout(0))

s_0 = (0, 1, 1, 1, 1, 1, 1)
t_0 = (-11, 10, 9, 11, 2, 1, 3)
(p_0,q_0) = (0, 2)


## 2. Unit moves and the four blocks — Appendix `app:z2-blocks`

For $X=(x_0,x_1,x_2,x_3)$ and $Y'=(y_0-1,y_1-1,y_2-1)$, set
\[
q_0=-e_4(X),\quad q_1=e_3(Y')-e_3(X),\quad
q_2=e_2(Y')-e_2(X),\quad q_3=e_1(Y')-e_1(X).
\]
The differential equation gives the last column of $\Theta(w)$ as $-(q_0,q_1,q_2)^T/q_3$. The upper increase is $I+\Theta(w)/x_i$; upper decreases and lower increases use the inverse formulas printed in the appendix. These are implemented in `theta` and `unit_move`.

For plateau $t=3$ or $4$, the four-block route is:

| Block | Direction, in paper order | Repetitions |
|---|---|---:|
| A | $(0,1,1,1;0,0,0)$ | $tn$ |
| B | $(0,0,0,0;2,1,3)$ | $n$ |
| C | $(0,10-t,9-t,11-t;0,0,0)$ | $n$ |
| D | $(-1,0,0,0;0,0,0)$ | $11n$ |

The following compares **all nine matrix entries**, using two paths and two implementations. Conservation is the reason these routes agree at every regular depth; the finite checks detect parameter-order and multiplication-order mistakes.

In [3]:
checks=[]
for n in (1,2,3):
    reference=z2.four_block_walk(n, plateau=3, backend="native")
    for plateau in (3,4):
        assert z2.four_block_walk(n, plateau, "native") == reference
        assert z2.four_block_walk(n, plateau, "ramanujantools") == reference
        checks.append((n, plateau, "all 9 entries agree"))
checks

[(1, 3, 'all 9 entries agree'),
 (1, 4, 'all 9 entries agree'),
 (2, 3, 'all 9 entries agree'),
 (2, 4, 'all 9 entries agree'),
 (3, 3, 'all 9 entries agree'),
 (3, 4, 'all 9 entries agree')]

## 3. Denominator normalization and the polynomial readout

Use **the same** $C_n$ as Lemma 6.20 (`prop:zeta2-int`):
\[
C_n=\prod_{u=9,10,11}\binom{un}{8n}
=\binom{10n}{2n}\binom{9n}{n}\binom{11n}{3n}.
\]
The appendix uses the polynomial $F(s)$ below; it is different from the special-function value $F_n$:
\[
F(s)=\prod_{u=9,10,11}\binom{s+un}{8n},\quad
c_j=(-1)^j\binom{11n}{j}F(j),\quad
S_0(s)=\sum_{j=0}^{11n}(-1)^j\binom{11n}{j}\frac{F(s)-F(j)}{s-j}.
\]
The quotient is a polynomial. At $s=j$ its value is $F'(j)$; no division by zero or floating-point limit is needed. The cancellation lowers $\deg S_0$ to at most $13n-1$.

For a polynomial $P$,
\[
J_\eta(P)=\sum_{h=0}^{\deg P}\frac{(-1)^h}{h+1}\Delta^hP(-\eta),
\qquad \Delta P(s)=P(s+1)-P(s).
\]
For every integer $2n\le\eta\le9n$, Appendix `app:z2-blocks` gives
\[
C_nq_n=2\sum_jc_j,\qquad
C_np_n=2\left(\sum_jc_j H^{(2)}_{j+\eta}+J_\eta(S_0)\right).
\]
Changing $\eta$ will later allow the arithmetic argument to choose a convenient shift for each prime.

In [4]:
rows=[]
for n in range(5):
    p_n,q_n=z2.matrix_readout(n)
    C_n=z2.normalization_C(n)
    for eta in sorted({2*n, 3*n, 9*n}):
        assert z2.closed_form_readout(n, eta) == (C_n*p_n, C_n*q_n)
    S0=z2.divided_sum(n)
    assert n == 0 or S0.degree() <= 13*n-1
    rows.append((n,C_n,S0.degree(),"readout and three shifts agree"))
rows

[(0, 1, -1, 'readout and three shifts agree'),
 (1, 66825, 12, 'readout and three shifts agree'),
 (2, 55309497705, 25, 'readout and three shifts agree'),
 (3, 66983025922312500, 38, 'readout and three shifts agree'),
 (4, 95542278495711936914025, 51, 'readout and three shifts agree')]

The next cell replays the supplied `check_finite_formula.py` from source. It checks 16 general vertices, the initial readout, the divided-sum identity, the $J_\eta$ shift relations, the companion relation, and each contiguity generator. This tests more than this one trajectory, but remains a finite diagnostic; the universal derivation is in the paper and supplied formalization.

In [5]:
certificate,seconds=z2.run_certificate("check_finite_formula.py")
assert certificate["initial_rows_match"]
assert certificate["state_count"] == 16
print(f"{certificate['state_count']} general vertices passed in {seconds}s.")

16 general vertices passed in 0.061s.


## 4. Full symbolic dual pairing — Lemma 6.17 (`lem:z2-connection`)

Write the forward differential operator as $P(\theta)-zR(\theta)$, where
$P(\theta)=\theta\prod_j(\theta+\beta_j-1)$ and
$R(\theta)=\prod_i(\theta+\alpha_i)$.
Its bilinear concomitant at $z=1$ gives a matrix $C(n)$ pairing the forward state with the direct adjoint Meijer state. The paper shifts the last upper Meijer parameter from $\alpha_3$ to $\alpha_3+1$. If $\Theta_G$ is the derivative matrix for this shifted Meijer equation, the contiguous relation is
$\mathcal H_n=\mathcal G_n A(n)$ with $A(n)=\alpha_3 I-\Theta_G(n)$.
Consequently $K(n)=C(n)A(n)^T$.

`dual_pairing.py` constructs this matrix and the two trajectory matrices from their parameter definitions. It verifies
\[
M(n)K(n+1)Q(n)^T=c(n)K(n),
\]
equivalently the paper's identity with $Q^\vee=Q^{-T}$. This is an identity in the rational-function field $\mathbb Q(n)$, not a check at selected depths. The determinant factors displayed below also show that $K(n)$ is invertible for every integer $n\ge0$.

In [6]:
from dual_pairing import certify_pairing
pairing=certify_pairing()
assert pairing["identity_verified"]
print("M(n) K(n+1) Q(n)^T = c(n) K(n): exact rational identity passed.")
print("det K(n) =",pairing["determinant_K"])
display(pairing["K"])
display(pairing["c"])
# The independent finite route has the same first forward step.
from flint import fmpq,fmpq_mat
M0=pairing["M"].subs({pairing["n"]:0})
M0_exact=fmpq_mat([[fmpq(str(M0[i,j])) for j in range(3)] for i in range(3)])
assert M0_exact == z2.four_block_walk(1)

M(n) K(n+1) Q(n)^T = c(n) K(n): exact rational identity passed.
det K(n) = -(8*n + 1)*(9*n + 1)*(10*n + 1)*(11*n + 1)*(13*n + 1)*(13*n + 2)


Matrix([
[n*(11*n + 1)*(1315*n**2 + 240*n + 11), -3*n*(11*n + 1)*(14*n + 1), -(11*n + 1)*(13*n + 1)],
[       (11*n + 1)*(42*n**2 - 8*n - 1),        101*n**2 + 32*n + 2,              -13*n - 1],
[               -(11*n + 1)*(13*n + 2),                   13*n + 2,                      0]])

(n + 1)/(10800*(5*n + 1)*(5*n + 2)*(5*n + 3)*(5*n + 4)*(9*n + 1)*(9*n + 2)*(9*n + 4)*(9*n + 5)*(9*n + 7)*(9*n + 8)*(10*n + 1)*(10*n + 3)*(10*n + 7)*(10*n + 9))

## 5. The initial dual pairing — Lemmas 6.17 and 6.18

With the paper's symbols,
\[
K(0)=\begin{pmatrix}0&0&-1\\-1&2&-1\\-2&2&0\end{pmatrix},\qquad
\mathcal G_0=(\zeta(2)-\zeta(3)-1,\zeta(2)-1,-1).
\]
The exact calculation below gives the limiting ratio $2\zeta(2)/2$. The Dual Convergence Theorem and recessiveness in Appendix `app:zeta2-proof` are the analytic inputs that make this the actual limit. This initial calculation alone does not prove convergence; the preceding symbolic calculation establishes the $n$-dependent dual pairing.

In [7]:
import sympy as sp
Z2,Z3=sp.symbols("zeta2 zeta3")
K0=sp.Matrix([[0,0,-1],[-1,2,-1],[-2,2,0]])
G0=sp.Matrix([Z2-Z3-1,Z2-1,-1])
rho_q=sp.Matrix([[2,0,0]])
rho_p=sp.Matrix([[0,2,-1]])
q_pair=(rho_q*K0*G0)[0]
p_pair=sp.expand((rho_p*K0*G0)[0])
assert K0.det() != 0 and q_pair == 2 and p_pair == 2*Z2
assert sp.simplify(p_pair/q_pair) == Z2
print("det K(0) =", K0.det())
print("denominator pairing =",q_pair,"; numerator pairing =",p_pair)

det K(0) = -2
denominator pairing = 2 ; numerator pairing = 2*zeta2


## 6. Numerical illustration of Lemma 6.18

We use 200 decimal digits so that cancellation in $\zeta(2)-p_n/q_n$ is resolved at the displayed depths. The rate in the third column is $-\log|\zeta(2)-p_n/q_n|/n$; finite-depth values need not equal the asymptotic $g$ of Lemma 6.19. Exact rate certificates are in notebook 03.

In [8]:
import mpmath as mp
mp.mp.dps=200
table=["| n | error | −log(error)/n |", "|---:|---:|---:|"]
for n in range(1,6):
    p_n,q_n=z2.matrix_readout(n)
    ratio=p_n/q_n
    approximation=mp.mpf(int(ratio.numerator))/int(ratio.denominator)
    error=abs(mp.zeta(2)-approximation)
    assert error>0
    table.append(f"| {n} | {mp.nstr(error,7)} | {mp.nstr(-mp.log(error)/n,10)} |")
display(Markdown("\n".join(table)))

| n | error | −log(error)/n |
|---:|---:|---:|
| 1 | 2.229472e-27 | 61.36803255 |
| 2 | 7.078773e-55 | 62.34253977 |
| 3 | 1.36024e-81 | 62.06724369 |
| 4 | 5.799811e-109 | 62.30598744 |
| 5 | 1.71781e-136 | 62.52210443 |

**What has been established computationally here:** exact agreement of the displayed CMF routes and RamanujanTools matrices at depths 1–3; closed-form readouts at depths 0–4 with three shifts; the supplied 16-vertex generator diagnostic; the full rational dual pairing for symbolic $n$; and its initial value. The next independent notebooks implement the prime arithmetic and exact scalar rate certificates. They do not require this notebook's outputs.